In [ ]:
import pandas as pd
from transformers import AutoTokenizer
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
INPUT_PATH = "Thuan_QA_data_evaluated.parquet"
TOKENIZER_MODEL = "vinai/bartpho-syllable"

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_MODEL)
df = pd.read_parquet(INPUT_PATH, engine="pyarrow")

In [ ]:
# Check answers that are "Thông tin không có trong văn bản" (before punctuation filter)
NO_INFO_PATTERNS = ["Thông tin không có trong văn bản"]

no_info_count = 0
no_info_assessments = {}

for _, row in df.iterrows():
    for qa in row['qa_pairs']:
        ans = qa['answer'].strip().rstrip('.')
        if ans in NO_INFO_PATTERNS:
            no_info_count += 1
            a = qa.get('overall_assessment', 'UNKNOWN')
            no_info_assessments[a] = no_info_assessments.get(a, 0) + 1

total_qa = sum(len(row['qa_pairs']) for _, row in df.iterrows())

print("=" * 60)
print(f'Answers matching: "{NO_INFO_PATTERNS[0]}"')
print("=" * 60)
print(f"Count: {no_info_count}/{total_qa} ({no_info_count/total_qa:.2%})")
print(f"\nAssessment breakdown:")
for a, c in sorted(no_info_assessments.items(), key=lambda x: -x[1]):
    print(f"  {a}: {c} ({c/no_info_count:.1%})" if no_info_count > 0 else f"  {a}: {c}")
print("=" * 60)

In [ ]:
# Filter: context/answer must end with '.', question must end with '?'
# - Bad context → remove entire row (all QA pairs)
# - Bad question/answer → remove only that QA pair, keep others
before_rows = len(df)
before_qa = sum(len(row['qa_pairs']) for _, row in df.iterrows())

# === 1. Detect & collect bad context rows ===
bad_context_mask = ~df['context'].str.strip().str.endswith('.')
bad_context_count = bad_context_mask.sum()
bad_context_qa_count = sum(len(row['qa_pairs']) for _, row in df[bad_context_mask].iterrows())
bad_context_samples = df[bad_context_mask]['context'].head(3).tolist()

# Assessment breakdown for bad-context QA pairs
bad_context_assessments = {}
for _, row in df[bad_context_mask].iterrows():
    for qa in row['qa_pairs']:
        a = qa.get('overall_assessment', 'UNKNOWN')
        bad_context_assessments[a] = bad_context_assessments.get(a, 0) + 1

# Remove bad context rows
df = df[~bad_context_mask].reset_index(drop=True)

# === 2. Detect & remove individual bad QA pairs (question/answer) ===
bad_question_samples, bad_answer_samples = [], []
bad_question_count, bad_answer_count = 0, 0
bad_question_assessments, bad_answer_assessments = {}, {}

for idx, row in df.iterrows():
    good_pairs = []
    for qa in row['qa_pairs']:
        q_ok = qa['question'].strip().endswith('?')
        a_ok = qa['answer'].strip().endswith('.')
        if q_ok and a_ok:
            good_pairs.append(qa)
        else:
            assess = qa.get('overall_assessment', 'UNKNOWN')
            if not q_ok:
                bad_question_count += 1
                bad_question_assessments[assess] = bad_question_assessments.get(assess, 0) + 1
                if len(bad_question_samples) < 3:
                    bad_question_samples.append(qa['question'])
            if not a_ok:
                bad_answer_count += 1
                bad_answer_assessments[assess] = bad_answer_assessments.get(assess, 0) + 1
                if len(bad_answer_samples) < 3:
                    bad_answer_samples.append(qa['answer'])
    df.at[idx, 'qa_pairs'] = good_pairs

# Remove rows with no QA pairs left
df = df[df['qa_pairs'].map(len) > 0].reset_index(drop=True)

after_rows = len(df)
after_qa = sum(len(row['qa_pairs']) for _, row in df.iterrows())

# === Report ===
print("=" * 70)
print("PUNCTUATION FILTER REPORT")
print("=" * 70)

print(f"\n[Context not ending with '.']  {bad_context_count} rows → {bad_context_qa_count} QA pairs removed")
print(f"  Assessment: {bad_context_assessments}")
for i, s in enumerate(bad_context_samples, 1):
    print(f"  Sample {i}: ...{s.strip()[-100:]}")

print(f"\n[Question not ending with '?'] {bad_question_count} QA pairs removed")
print(f"  Assessment: {bad_question_assessments}")
for i, s in enumerate(bad_question_samples, 1):
    print(f"  Sample {i}: {s.strip()[:120]}")

print(f"\n[Answer not ending with '.']   {bad_answer_count} QA pairs removed")
print(f"  Assessment: {bad_answer_assessments}")
for i, s in enumerate(bad_answer_samples, 1):
    print(f"  Sample {i}: {s.strip()[:120]}")

# Combined summary
all_assessments = {}
for d in [bad_context_assessments, bad_question_assessments, bad_answer_assessments]:
    for k, v in d.items():
        all_assessments[k] = all_assessments.get(k, 0) + v

print(f"\n--- Total Removed Assessment Breakdown ---")
for a, c in sorted(all_assessments.items(), key=lambda x: -x[1]):
    print(f"  {a}: {c}")

print(f"\n--- Overall Summary ---")
print(f"Rows:     {after_rows:,}/{before_rows:,} ({before_rows - after_rows:,} removed)")
print(f"QA pairs: {after_qa:,}/{before_qa:,} ({before_qa - after_qa:,} removed, {(before_qa - after_qa)/before_qa:.2%})")
print("=" * 70)

# 1. Basic statistics

In [ ]:
def display_schema(df):
    sample = df.iloc[0]
    for col in df.columns:
        col_dtype = type(sample[col])
        print(f">- Name: {col}\t- Dtype: {col_dtype}")
        if col == 'qa_pairs':
            qa_pair = sample[col][0]
            main_attributes = ("question", "answer", "topic_id")
            for key, value in qa_pair.items():
                label = "(main)" if key in main_attributes else ""
                print(f"  >- Name: {key}\tDtype: {type(value)} {label}")

In [ ]:
# split topic_id from question and filter out qa_pairs without topic_id
import re
def extract_topic_id(question):
    pattern = re.compile(r'^(FACTOID|SUMMARY|COMPARISON|VERIFICATION)-(.+)')
    match = pattern.match(question)
    if match:
        return match.group(1)
    else:
        return None

num_rows = len(df)
num_qa_pairs = sum(len(row['qa_pairs']) for _, row in df.iterrows())

# Add topic_id to qa_pairs and filter out those without topic_id
for idx, row in df.iterrows():
    for qa in row['qa_pairs']:
        topic_id = extract_topic_id(qa['question'])
        qa['topic_id'] = topic_id
    # Filter out qa_pairs where topic_id is None
    df.at[idx, 'qa_pairs'] = [qa for qa in row['qa_pairs'] if qa.get('topic_id') is not None]

# Remove rows that have no qa_pairs left
df = df[df['qa_pairs'].map(len) > 0].reset_index(drop=True)

_num_qa_pairs = [len(row['qa_pairs']) for _, row in df.iterrows()]

removed_rows_count = num_rows - len(df)
removed_qa_pairs_count = num_qa_pairs - sum(_num_qa_pairs)

################# Display stats after cleaning #################
print("===== After basic empty removal =====")
print(f"Total rows (context): {len(df)}/{num_rows}, {removed_rows_count:,} ({removed_rows_count / num_rows:.2%}) removed.")
print(f"Total QA pairs: {sum(_num_qa_pairs)}/{num_qa_pairs}, {removed_qa_pairs_count:,} ({removed_qa_pairs_count / num_qa_pairs:.2%}) removed")

print("=" * 40)
print("QA pairs per row analysis:")
print("\tMean: ", np.mean(_num_qa_pairs))
print("\tMedian: ", np.median(_num_qa_pairs))
print("\tStd Dev: ", np.std(_num_qa_pairs))
print("\tMin: ", np.min(_num_qa_pairs))
print("\tMax: ", np.max(_num_qa_pairs))
print("\t25%: ", np.percentile(_num_qa_pairs, 25))
print("\t50%: ", np.percentile(_num_qa_pairs, 50))
print("\t75%: ", np.percentile(_num_qa_pairs, 75))
print("\t90%: ", np.percentile(_num_qa_pairs, 90))
print("=" * 40)

print("Schema:")
display_schema(df)

In [ ]:
# Flatten QA pairs for statistics
dedupe_context = True

meta_cols = ['url', 'title', 'time']
qa_fields = [
    'question', 'answer', 'topic_id',
    'overall_assessment', 'overall_reason', 'total_score',
    'answerability_score', 'answerability_reason',
    'answer_accuracy_score', 'answer_accuracy_reason',
    'clarity_score', 'clarity_reason',
    'conciseness_score', 'conciseness_reason',
    'usefulness_score', 'usefulness_reason'
]

df_rows = df[meta_cols + ['context', 'qa_pairs']].copy()
df_rows = df_rows[df_rows['context'].notna()]

if dedupe_context:
    df_rows = df_rows.drop_duplicates(subset=['context']).reset_index(drop=True)

df_qa = df_rows.explode('qa_pairs', ignore_index=True)
df_qa = df_qa[df_qa['qa_pairs'].notna()]
qa_norm = pd.json_normalize(df_qa['qa_pairs'])
df_qa = pd.concat([df_qa.drop(columns=['qa_pairs']), qa_norm], axis=1)

df_flat = df_qa[meta_cols + ['context'] + qa_fields].reset_index(drop=True)

total_evaluated = len(df_flat)
# Get unique assessment labels from data
assessment_labels = df_flat['overall_assessment'].unique().tolist()
print(f"Assessment labels found: {assessment_labels}")

# 2. Token Length Analysis

In [ ]:
df_flat['context_tokens'] = df_flat['context'].apply(lambda x: len(tokenizer(x)['input_ids']))
df_flat['question_tokens'] = df_flat['question'].apply(lambda x: len(tokenizer(x)['input_ids']))
df_flat['answer_tokens'] = df_flat['answer'].apply(lambda x: len(tokenizer(x)['input_ids']))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Context tokens distribution
axes[0, 0].hist(df_flat['context_tokens'], bins=50, edgecolor='black')
axes[0, 0].set_title('Context Token Length Distribution')
axes[0, 0].set_xlabel('Number of Tokens')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].axvline(np.mean(df_flat['context_tokens']), color='r', linestyle='--', label=f'Mean: {np.mean(df_flat["context_tokens"]):.1f}')
axes[0, 0].axvline(np.percentile(df_flat['context_tokens'], 50), color='g', linestyle='--', label=f'Median: {np.percentile(df_flat["context_tokens"], 50):.1f}')
axes[0, 0].legend()

# Question tokens distribution
axes[0, 1].hist(df_flat['question_tokens'], bins=50, edgecolor='black')
axes[0, 1].set_title('Question Token Length Distribution')
axes[0, 1].set_xlabel('Number of Tokens')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].axvline(np.mean(df_flat['question_tokens']), color='r', linestyle='--', label=f'Mean: {np.mean(df_flat["question_tokens"]):.1f}')
axes[0, 1].axvline(np.percentile(df_flat['question_tokens'], 50), color='g', linestyle='--', label=f'Median: {np.percentile(df_flat["question_tokens"], 50):.1f}')
axes[0, 1].legend()

# Answer tokens distribution
axes[1, 0].hist(df_flat['answer_tokens'], bins=50, edgecolor='black')
axes[1, 0].set_title('Answer Token Length Distribution')
axes[1, 0].set_xlabel('Number of Tokens')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].axvline(np.mean(df_flat['answer_tokens']), color='r', linestyle='--', label=f'Mean: {np.mean(df_flat["answer_tokens"]):.1f}')
axes[1, 0].axvline(np.percentile(df_flat['answer_tokens'], 50), color='g', linestyle='--', label=f'Median: {np.percentile(df_flat["answer_tokens"], 50):.1f}')
axes[1, 0].legend()

# Summary statistics table
axes[1, 1].axis('tight')
axes[1, 1].axis('off')
stats_data = [
    ['Metric', 'Context', 'Question', 'Answer'],
    ['Mean', f'{np.mean(df_flat["context_tokens"]):.1f}', f'{np.mean(df_flat["question_tokens"]):.1f}', f'{np.mean(df_flat["answer_tokens"]):.1f}'],
    ['Median', f'{np.median(df_flat["context_tokens"]):.1f}', f'{np.median(df_flat["question_tokens"]):.1f}', f'{np.median(df_flat["answer_tokens"]):.1f}'],
    ['Min', f'{np.min(df_flat["context_tokens"])}', f'{np.min(df_flat["question_tokens"])}', f'{np.min(df_flat["answer_tokens"])}'],
    ['Max', f'{np.max(df_flat["context_tokens"])}', f'{np.max(df_flat["question_tokens"])}', f'{np.max(df_flat["answer_tokens"])}'],
    ['Std Dev', f'{np.std(df_flat["context_tokens"]):.1f}', f'{np.std(df_flat["question_tokens"]):.1f}', f'{np.std(df_flat["answer_tokens"]):.1f}'],
    ['15%', f'{np.percentile(df_flat["context_tokens"], 15):.1f}', f'{np.percentile(df_flat["question_tokens"], 15):.1f}', f'{np.percentile(df_flat["answer_tokens"], 15):.1f}'],
    ['25%', f'{np.percentile(df_flat["context_tokens"], 25):.1f}', f'{np.percentile(df_flat["question_tokens"], 25):.1f}', f'{np.percentile(df_flat["answer_tokens"], 25):.1f}'],
    ['50%', f'{np.percentile(df_flat["context_tokens"], 50):.1f}', f'{np.percentile(df_flat["question_tokens"], 50):.1f}', f'{np.percentile(df_flat["answer_tokens"], 50):.1f}'],
    ['75%', f'{np.percentile(df_flat["context_tokens"], 75):.1f}', f'{np.percentile(df_flat["question_tokens"], 75):.1f}', f'{np.percentile(df_flat["answer_tokens"], 75):.1f}'],
    ['95%', f'{np.percentile(df_flat["context_tokens"], 95):.1f}', f'{np.percentile(df_flat["question_tokens"], 95):.1f}', f'{np.percentile(df_flat["answer_tokens"], 95):.1f}']
]
table = axes[1, 1].table(cellText=stats_data, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.8)

plt.tight_layout()
plt.show()

In [ ]:
# Inspect outliers for context tokens
print("=" * 60)
print("CONTEXT TOKEN OUTLIERS")
print("=" * 60)

# Find min and max context token indices
min_context_idx = np.argmin(df_flat['context_tokens'])
max_context_idx = np.argmax(df_flat['context_tokens'])

print(f"\n[Minimum Context Tokens: {df_flat['context_tokens'].iloc[min_context_idx]}]")
print(f"Context text:\n{df_flat['context'].iloc[min_context_idx]}\n")

print(f"[Maximum Context Tokens: {df_flat['context_tokens'].iloc[max_context_idx]}]")
print(f"Context text:\n{df_flat['context'].iloc[max_context_idx][:500]}...\n")  # Show first 500 chars

# Inspect outliers for questions
print("=" * 60)
print("QUESTION TOKEN OUTLIERS")
print("=" * 60)

min_q_idx = df_flat['question_tokens'].idxmin()
max_q_idx = df_flat['question_tokens'].idxmax()

print(f"\n[Minimum Question Tokens: {df_flat.loc[min_q_idx, 'question_tokens']}]")
print(f"Question: {df_flat.loc[min_q_idx, 'question']}\n")

print(f"[Maximum Question Tokens: {df_flat.loc[max_q_idx, 'question_tokens']}]")
print(f"Question: {df_flat.loc[max_q_idx, 'question']}\n")

# Inspect outliers for answers
print("=" * 60)
print("ANSWER TOKEN OUTLIERS")
print("=" * 60)

min_a_idx = df_flat['answer_tokens'].idxmin()
max_a_idx = df_flat['answer_tokens'].idxmax()

print(f"\n[Minimum Answer Tokens: {df_flat.loc[min_a_idx, 'answer_tokens']}]")
print(f"Answer: {df_flat.loc[min_a_idx, 'answer']}\n")

print(f"[Maximum Answer Tokens: {df_flat.loc[max_a_idx, 'answer_tokens']}]")
print(f"Answer: {df_flat.loc[max_a_idx, 'answer']}\n")

print("=" * 60)

In [ ]:
# Trimming
MAX_CONTEXT_TOKENS = 1300

MAX_QUESTION_TOKENS = 100 
MIN_QUESTION_TOKENS = 11

MAX_ANSWER_TOKENS = 200 
MIN_ANSWER_TOKENS = 5

trim_mask = (
    (df_flat['context_tokens'] <= MAX_CONTEXT_TOKENS) &
    (df_flat['question_tokens'].between(MIN_QUESTION_TOKENS, MAX_QUESTION_TOKENS)) &
    (df_flat['answer_tokens'].between(MIN_ANSWER_TOKENS, MAX_ANSWER_TOKENS))
)

df_trimmed = df_flat[trim_mask].reset_index(drop=True)

print(f"Before: {len(df_flat):,}")
print(f"After:  {len(df_trimmed):,}")
print(f"Removed: {len(df_flat) - len(df_trimmed):,} ({(1 - len(df_trimmed)/len(df_flat)):.2%})")

# 3. Topic Distribution

In [ ]:
topic_ids = df_trimmed['topic_id'].value_counts().index.tolist()
topic_counts = df_trimmed['topic_id'].value_counts().values.tolist()

# Plot pie chart
plt.figure(figsize=(6, 6))
plt.pie(topic_counts, labels=topic_ids, autopct='%1.1f%%', startangle=140)
plt.title('Topic Distribution after Trimming')
plt.axis('equal')  # Equal aspect ratio ensures that pie is drawn as a circle.
plt.show()

# 4. Evaluation Score Distribution

In [ ]:
score_cols = ['answerability_score', 'answer_accuracy_score', 'clarity_score', 'conciseness_score', 'usefulness_score']

# Convert score columns to numeric
for col in score_cols:
    df_trimmed[col] = pd.to_numeric(df_trimmed[col], errors='coerce')
df_trimmed['total_score'] = pd.to_numeric(df_trimmed['total_score'], errors='coerce')

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(score_cols):
    data = df_trimmed[col].dropna()
    axes[i].hist(data, bins=20, edgecolor='black', color='steelblue', alpha=0.8)
    axes[i].set_title(col.replace('_', ' ').title(), fontsize=12)
    axes[i].set_xlabel('Score')
    axes[i].set_ylabel('Frequency')
    axes[i].axvline(data.mean(), color='r', linestyle='--', label=f'Mean: {data.mean():.2f}')
    axes[i].axvline(data.median(), color='g', linestyle='--', label=f'Median: {data.median():.2f}')
    axes[i].legend(fontsize=9)

# Total score distribution
data = df_trimmed['total_score'].dropna()
axes[5].hist(data, bins=30, edgecolor='black', color='darkorange', alpha=0.8)
axes[5].set_title('Total Score', fontsize=12)
axes[5].set_xlabel('Score')
axes[5].set_ylabel('Frequency')
axes[5].axvline(data.mean(), color='r', linestyle='--', label=f'Mean: {data.mean():.2f}')
axes[5].axvline(data.median(), color='g', linestyle='--', label=f'Median: {data.median():.2f}')
axes[5].legend(fontsize=9)

plt.suptitle('Evaluation Score Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Summary statistics table
print("=" * 80)
print("SCORE SUMMARY STATISTICS")
print("=" * 80)
score_stats = df_trimmed[score_cols + ['total_score']].describe().round(2)
print(score_stats.to_string())
print("=" * 80)

# 5. Quality Assessment Overview

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Overall assessment distribution
assessment_counts = df_trimmed['overall_assessment'].value_counts()
colors_map = {'ACCEPT': '#2ecc71', 'REVISE': '#e74c3c'}
colors = [colors_map.get(label, '#3498db') for label in assessment_counts.index]
axes[0].bar(assessment_counts.index, assessment_counts.values, color=colors, edgecolor='black')
for i, (label, count) in enumerate(zip(assessment_counts.index, assessment_counts.values)):
    axes[0].text(i, count + len(df_trimmed)*0.005, f'{count}\n({count/len(df_trimmed):.1%})', ha='center', fontsize=10)
axes[0].set_title('Overall Assessment Distribution', fontsize=12)
axes[0].set_ylabel('Count')

# Accept rate by topic_id
accept_by_topic = df_trimmed.groupby('topic_id')['overall_assessment'].apply(
    lambda x: (x == 'ACCEPT').sum() / len(x) * 100
).sort_values(ascending=False)
bars = axes[1].bar(accept_by_topic.index, accept_by_topic.values, color='steelblue', edgecolor='black')
for bar, val in zip(bars, accept_by_topic.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f'{val:.1f}%', ha='center', fontsize=10)
axes[1].set_title('Accept Rate by Topic', fontsize=12)
axes[1].set_ylabel('Accept Rate (%)')
axes[1].set_ylim(0, 105)

# Score box plot by overall assessment
assessment_groups = df_trimmed['overall_assessment'].unique()
if len(assessment_groups) > 0:
    data_by_group = [df_trimmed[df_trimmed['overall_assessment'] == g]['total_score'].dropna().values for g in assessment_groups]
    # Filter out empty groups
    valid = [(g, d) for g, d in zip(assessment_groups, data_by_group) if len(d) > 0]
    if valid:
        groups, data = zip(*valid)
        bp = axes[2].boxplot(data, labels=groups, patch_artist=True)
        box_colors = ['#2ecc71', '#e74c3c']
        for patch, color in zip(bp['boxes'], box_colors[:len(groups)]):
            patch.set_facecolor(color)
            patch.set_alpha(0.6)
axes[2].set_title('Total Score by Assessment', fontsize=12)
axes[2].set_xlabel('Overall Assessment')
axes[2].set_ylabel('Total Score')

plt.tight_layout()
plt.show()

# 6. Score Correlation Analysis

In [ ]:
import seaborn as sns

corr_cols = score_cols + ['total_score', 'context_tokens', 'question_tokens', 'answer_tokens']
corr_matrix = df_trimmed[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdYlBu_r',
            center=0, vmin=-1, vmax=1, square=True,
            linewidths=0.5, ax=ax,
            xticklabels=[c.replace('_', '\n') for c in corr_cols],
            yticklabels=[c.replace('_', '\n') for c in corr_cols])
ax.set_title('Correlation Heatmap: Scores & Token Lengths', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# 7. Topic-wise Score Analysis

In [ ]:
# Grouped bar chart: mean score per dimension by topic
topic_score_means = df_trimmed.groupby('topic_id')[score_cols].mean()

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(topic_score_means.index))
width = 0.15
colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6']

for i, col in enumerate(score_cols):
    bars = ax.bar(x + i * width, topic_score_means[col], width, label=col.replace('_score', '').title(), color=colors[i], edgecolor='black', alpha=0.85)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=7)

ax.set_xlabel('Topic ID')
ax.set_ylabel('Mean Score')
ax.set_title('Mean Scores by Topic and Dimension', fontsize=13, fontweight='bold')
ax.set_xticks(x + width * 2)
ax.set_xticklabels(topic_score_means.index)
ax.legend(loc='lower right', fontsize=9)
ax.set_ylim(0, ax.get_ylim()[1] * 1.1)
plt.tight_layout()
plt.show()

# Table view
print("=" * 80)
print("MEAN SCORES BY TOPIC")
print("=" * 80)
topic_summary = df_trimmed.groupby('topic_id').agg(
    count=('total_score', 'size'),
    total_score_mean=('total_score', 'mean'),
    total_score_std=('total_score', 'std'),
    **{f'{col}_mean': (col, 'mean') for col in score_cols}
).round(2)
print(topic_summary.to_string())
print("=" * 80)

In [ ]:
# Box plot: total score distribution by topic
fig, ax = plt.subplots(figsize=(10, 6))
topics = df_trimmed['topic_id'].unique()
data_by_topic = [df_trimmed[df_trimmed['topic_id'] == t]['total_score'].dropna().values for t in topics]

bp = ax.boxplot(data_by_topic, labels=topics, patch_artist=True, notch=True)
colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']
for patch, color in zip(bp['boxes'], colors[:len(topics)]):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)

ax.set_title('Total Score Distribution by Topic', fontsize=13, fontweight='bold')
ax.set_xlabel('Topic')
ax.set_ylabel('Total Score')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# 8. Token Length vs Score Relationship

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

token_cols = ['context_tokens', 'question_tokens', 'answer_tokens']
titles = ['Context Length vs Total Score', 'Question Length vs Total Score', 'Answer Length vs Total Score']

for i, (tcol, title) in enumerate(zip(token_cols, titles)):
    valid = df_trimmed[[tcol, 'total_score']].dropna()
    axes[i].scatter(valid[tcol], valid['total_score'], alpha=0.2, s=10, color='steelblue')
    # Trend line
    z = np.polyfit(valid[tcol], valid['total_score'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(valid[tcol].min(), valid[tcol].max(), 100)
    axes[i].plot(x_line, p(x_line), 'r--', linewidth=2, label=f'Trend (slope={z[0]:.4f})')
    axes[i].set_title(title, fontsize=12)
    axes[i].set_xlabel('Token Count')
    axes[i].set_ylabel('Total Score')
    axes[i].legend(fontsize=9)
    axes[i].grid(alpha=0.3)

plt.suptitle('Token Length vs Total Score', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# 9. Data Quality & Duplicate Analysis

In [ ]:
print("=" * 60)
print("DATA QUALITY REPORT")
print("=" * 60)

# Missing values
print("\n--- Missing Values ---")
missing = df_trimmed[['context', 'question', 'answer', 'topic_id', 'total_score', 'overall_assessment']].isnull().sum()
for col, count in missing.items():
    print(f"  {col}: {count} ({count/len(df_trimmed):.2%})")

# Duplicate questions
dup_questions = df_trimmed['question'].duplicated().sum()
print(f"\n--- Duplicates ---")
print(f"  Duplicate questions: {dup_questions} ({dup_questions/len(df_trimmed):.2%})")

# Duplicate answers
dup_answers = df_trimmed['answer'].duplicated().sum()
print(f"  Duplicate answers: {dup_answers} ({dup_answers/len(df_trimmed):.2%})")

# Duplicate (question, answer) pairs
dup_qa = df_trimmed.duplicated(subset=['question', 'answer']).sum()
print(f"  Duplicate QA pairs: {dup_qa} ({dup_qa/len(df_trimmed):.2%})")

# Unique contexts
unique_contexts = df_trimmed['context'].nunique()
print(f"\n--- Unique Counts ---")
print(f"  Unique contexts: {unique_contexts}")
print(f"  Unique questions: {df_trimmed['question'].nunique()}")
print(f"  Unique answers: {df_trimmed['answer'].nunique()}")
print(f"  QA pairs per unique context: {len(df_trimmed)/unique_contexts:.2f}")

# Empty or very short text checks
print(f"\n--- Short Text Check ---")
short_q = (df_trimmed['question'].str.len() < 10).sum()
short_a = (df_trimmed['answer'].str.len() < 5).sum()
print(f"  Questions < 10 chars: {short_q}")
print(f"  Answers < 5 chars: {short_a}")

print("=" * 60)

# 10. Low-score Sample Inspection

In [ ]:
# Inspect worst-scoring QA pairs
n_samples = 5
worst = df_trimmed.nsmallest(n_samples, 'total_score')[['topic_id', 'question', 'answer', 'total_score', 'overall_assessment', 'overall_reason']]

print(f"===== Bottom {n_samples} QA Pairs by Total Score =====\n")
for i, (_, row) in enumerate(worst.iterrows(), 1):
    print(f"--- Sample {i} (score={row['total_score']}, assessment={row['overall_assessment']}) ---")
    print(f"  Topic: {row['topic_id']}")
    print(f"  Q: {row['question'][:200]}")
    print(f"  A: {row['answer'][:200]}")
    print(f"  Reason: {row['overall_reason'][:300]}")
    print()

# Inspect best-scoring QA pairs
best = df_trimmed.nlargest(n_samples, 'total_score')[['topic_id', 'question', 'answer', 'total_score', 'overall_assessment', 'overall_reason']]

print(f"===== Top {n_samples} QA Pairs by Total Score =====\n")
for i, (_, row) in enumerate(best.iterrows(), 1):
    print(f"--- Sample {i} (score={row['total_score']}, assessment={row['overall_assessment']}) ---")
    print(f"  Topic: {row['topic_id']}")
    print(f"  Q: {row['question'][:200]}")
    print(f"  A: {row['answer'][:200]}")
    print(f"  Reason: {row['overall_reason'][:300]}")
    print()

# 11. Final Dataset Summary

In [ ]:
print("=" * 60)
print("FINAL DATASET SUMMARY")
print("=" * 60)

accept_count = (df_trimmed['overall_assessment'] == 'ACCEPT').sum()
revise_count = (df_trimmed['overall_assessment'] == 'REVISE').sum()

print(f"Total QA pairs (after trimming): {len(df_trimmed):,}")
print(f"Unique contexts: {df_trimmed['context'].nunique():,}")
print(f"Topics: {', '.join(df_trimmed['topic_id'].unique())}")
print()
print(f"Assessment breakdown:")
print(f"  ACCEPT: {accept_count:,} ({accept_count/len(df_trimmed):.1%})")
print(f"  REVISE: {revise_count:,} ({revise_count/len(df_trimmed):.1%})")
print()
print(f"Token length ranges (after trimming):")
print(f"  Context:  {df_trimmed['context_tokens'].min()} - {df_trimmed['context_tokens'].max()} (mean: {df_trimmed['context_tokens'].mean():.0f})")
print(f"  Question: {df_trimmed['question_tokens'].min()} - {df_trimmed['question_tokens'].max()} (mean: {df_trimmed['question_tokens'].mean():.0f})")
print(f"  Answer:   {df_trimmed['answer_tokens'].min()} - {df_trimmed['answer_tokens'].max()} (mean: {df_trimmed['answer_tokens'].mean():.0f})")
print()
print(f"Mean total score: {df_trimmed['total_score'].mean():.2f} ± {df_trimmed['total_score'].std():.2f}")
print("=" * 60)

In [ ]:
df_trimmed.head()

In [ ]:
# Keep only ACCEPT rows, selected columns, and rename topic_id -> question_type
df_accept = (
    df_trimmed.loc[df_trimmed['overall_assessment'] == 'ACCEPT', ['url', 'title', 'time', 'context', 'question', 'answer', 'topic_id']]
    .rename(columns={'topic_id': 'question_type'})
    .reset_index(drop=True)
)

print(f"Rows after filtering ACCEPT: {len(df_accept):,}")
print(f"Columns: {list(df_accept.columns)}")
df_accept.head()

In [ ]:
# Remove leading question-type prefix such as FACTOID-, SUMMARY-, ... from question text
df_accept['question'] = df_accept['question'].str.replace(r'^\s*[A-Z_]+\s*-\s*', '', regex=True)

df_accept.head()

In [ ]:
df_accept.to_csv('QA_dataset_final_Thuan.csv', index=False)